# Fase 3: Data cleaning

In [1]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Fase 3: Data Cleaning

En esta fase se prepara una copia del dataset para trabajar, se eliminan las variables que no aportan información y se comprueba la calidad de los datos restantes.

In [2]:
df_original = pd.read_csv("../data/ai4i2020.csv")
print(f"Dimensiones del dataset original: {df_original.shape[0]} filas y {df_original.shape[1]} columnas.")

Dimensiones del dataset original: 10000 filas y 14 columnas.



## 3.1 Copia de trabajo y eliminación de identificadores

`UDI` y `Product ID` son identificadores únicos de cada registro (10.000 valores únicos en 10.000 filas), por lo que no aportan información para el análisis ni para un futuro modelo. Se eliminan trabajando sobre una copia, para conservar el dataset original.

In [3]:
df = df_original.copy()
df = df.drop(columns=["UDI", "Product ID"])

print(f"Dimensiones tras eliminar identificadores: {df.shape[0]} filas y {df.shape[1]} columnas.")

Dimensiones tras eliminar identificadores: 10000 filas y 12 columnas.


In [4]:
# Comprobación de duplicados
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 0


In [5]:
# Revisión de la columna Type
print(df["Type"].value_counts())

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64


### Conclusión 3.2: duplicados

No hay filas duplicadas una vez eliminados los identificadores (0 duplicados), así que no hace falta eliminar ningún registro.

### Conclusión 3.3: variable `Type`

`Type` tiene exactamente 3 categorías (L, M y H), sin errores de escritura ni valores inesperados. La distribución no es uniforme:

- **L:** 6.000 registros (60 %)
- **M:** 2.997 registros (aprox. 30 %)
- **H:** 1.003 registros (aprox. 10 %)

La categoría L es la mayoritaria, por lo que habrá que tenerlo en cuenta al analizar la relación de `Type` con los fallos. No requiere limpieza.

## 3.4 Fuga de datos (data leakage)

`TWF`, `HDF`, `PWF`, `OSF` y `RNF` indican el tipo de fallo, información que solo se conoce cuando el fallo ya ha ocurrido. Antes de eliminarlas, se comprueba su relación con la variable objetivo `Machine failure`.

In [6]:
# Lista de columnas que indican el tipo de fallo
fallos = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# Creamos una columna auxiliar "algun_fallo":
# vale 1 si la máquina tiene al menos un tipo de fallo marcado, y 0 si no tiene ninguno.
# (max(axis=1) toma el valor más alto de cada fila entre las 5 columnas)
df["algun_fallo"] = df[fallos].max(axis=1)

# Tabla de contingencia: cuenta cuántas filas hay en cada combinación de
# "algun_fallo" (filas) y "Machine failure" (columnas).
# Si ambas columnas coinciden casi siempre, hay fuga de datos.
print(pd.crosstab(df["algun_fallo"], df["Machine failure"]))

Machine failure     0    1
algun_fallo               
0                9643    9
1                  18  330


In [7]:
# Filtramos las filas "incoherentes": aquellas donde lo que dice "Machine failure"
# NO coincide con lo que dicen las columnas de tipo de fallo (algun_fallo).
# Es decir, hay fallo sin tipo registrado, o tipo de fallo registrado sin fallo.
incoherentes = df[df["algun_fallo"] != df["Machine failure"]]

# Sumamos cada columna solo en esas filas incoherentes para ver de dónde vienen:
# - Si "Machine failure" suma algo, hay fallos sin ningún tipo registrado.
# - Si alguna columna de tipo de fallo suma algo (p. ej. RNF), hay un tipo de fallo
#   marcado pero "Machine failure" vale 0.
print(incoherentes[["Machine failure"] + fallos].sum())

Machine failure     9
TWF                 0
HDF                 0
PWF                 0
OSF                 0
RNF                18
dtype: int64


In [8]:
# Nos quedamos solo con las filas donde RNF = 0.

# Hay que hacerlo ANTES de borrar la columna RNF, porque si no ya no podríamos filtrar por ella.
df = df[df["RNF"] == 0].copy()

# Eliminamos las columnas de tipo de fallo (fuga de datos) y la columna auxiliar
df = df.drop(columns=fallos + ["algun_fallo"])

print(f"Dimensiones tras la limpieza: {df.shape[0]} filas y {df.shape[1]} columnas.")
print(df.columns.tolist())

Dimensiones tras la limpieza: 9981 filas y 7 columnas.
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


### Conclusión 3.4: fuga de datos

`Machine failure` coincide con la existencia de algún tipo de fallo en 9.973 de las 10.000 filas (99,7 %). Las columnas `TWF`, `HDF`, `PWF`, `OSF` y `RNF` revelan prácticamente el valor del objetivo y solo se conocen una vez que el fallo ya ha ocurrido, por lo que provocarían fuga de datos.

Hay 27 registros incoherentes:
- **18 filas** con `RNF = 1` y `Machine failure = 0`.
- **9 filas** con `Machine failure = 1` sin ningún tipo de fallo registrado.

`TWF`, `HDF`, `PWF` y `OSF` son coherentes con `Machine failure` en todos los registros.

### Decisión

- Se eliminan las 19 filas con `RNF = 1`: es un fallo aleatorio, independiente de las variables del proceso, por lo que no puede predecirse a partir de ellas. De esas filas, 18 eran además incoherentes (`Machine failure = 0`).
- Se mantienen las 9 filas con `Machine failure = 1` sin causa registrada, porque son fallos reales y hay pocos en el dataset.
- Se eliminan las columnas de tipo de fallo y la auxiliar `algun_fallo` para evitar fuga de datos.

Tras la limpieza, el dataset queda con 9.981 filas y 7 columnas: `Type`, las cinco variables numéricas y `Machine failure`.

In [9]:
# Guardamos el dataset limpio en una nueva variable
df_clean = df.copy()

# Lo guardamos también como archivo CSV para poder usarlo en el resto del proyecto
# (index=False evita que pandas añada una columna extra con el índice)
df_clean.to_csv("../data/ai4i2020_clean.csv", index=False)

print(f"Dataset limpio guardado: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas.")

Dataset limpio guardado: 9981 filas y 7 columnas.
